# Notebook 3: Graph Context To Generated Code

This notebook shows a transparent generation flow rather than a black-box demo. It starts with graph context, reads the sample configuration, and then produces a public-safe Java artifact that can be reviewed and refined.

The emphasis is on explainability: what came from the graph, what came from configuration, and what assumptions shape the generated output.

In [ ]:
import os
import sys
from pathlib import Path

import toml
from dotenv import load_dotenv

env_path = Path('..') / '.env'
load_dotenv(dotenv_path=env_path)
sys.path.append(str(Path('../src').resolve()))

from neo4j_agent import Neo4jAgent

agent = Neo4jAgent(
    uri=os.getenv('NEO4J_URI', 'bolt://localhost:7687'),
    user=os.getenv('NEO4J_USER', 'neo4j'),
    password=os.getenv('NEO4J_PASSWORD', 'password')
)

status = agent.verify_connection()
print(f'Using environment file: {env_path}')
print(f'Connected: {status["connected"]}')

## Step 1: Pull Structural Context From The Graph

A generation workflow is stronger when it begins with project structure instead of guesses. This query stands in for the kind of version-scoped lookup you would typically expose through a small MCP toolset.

In [ ]:
query = """
MATCH (c:Class)
WHERE c.gitTag = $version
OPTIONAL MATCH (c)-[:DEFINES_METHOD]->(m:Method)
RETURN c.name AS class_name, c.category AS category, collect(DISTINCT m.name) AS methods
"""

results = agent.query(query, {'version': 'v2.3.0'})
for record in results:
    print(record)

## Step 2: Read The Runtime Configuration

The graph tells you what exists in the modeled system. The configuration file tells you what a particular module wants to do at runtime. Generation quality depends on combining both sources cleanly.

In [ ]:
toml_path = Path('../examples/sample_project/Example.toml')
config = toml.load(toml_path)
conditions = config['example_module']['ConditionsAndGradeables']
test_case_config = config['example_module']['test1']['max']

print('Module: example_module')
print('Conditions:', conditions['testConditions'])
print('End condition:', conditions['endCondition'])
print('Groups:', conditions['gradeableLists'])
print('')
for key, value in test_case_config.items():
    print(f'{key}: {value}')

## Step 3: Generate A Public-Safe Java Artifact

The generated class below uses neutral framework names. That keeps the notebook useful as a reference while avoiding internal package conventions or product-specific naming.

In [ ]:
def generate_test_method(module_name):
    class_name = 'ExampleGeneratedTestMethod'
    lines = [
        'package testmethod;',
        '',
        'import framework.actions.LevelChangeAction;',
        'import framework.config.ConfigBlock;',
        'import framework.config.ConfigLoader;',
        'import framework.runtime.BaseTestMethod;',
        'import framework.runtime.TestCaseBase;',
        'import framework.runtime.TestList;',
        'import framework.runtime.TestListManager;',
        'import java.util.List;',
        '',
        '/**',
        f' * Generated example test method for {module_name}.',
        ' */',
        f'public class {class_name} extends BaseTestMethod ' + '{',
        '  @Override',
        '  protected void defineTestSequences(TestListManager testListManager) {',
        '    String paramFile = "testtables/Example.toml";',
        f'    ConfigBlock config = ConfigLoader.load(paramFile, "{module_name}.ConditionsAndGradeables");',
        '',
        '    String endCondition = config.getString("endCondition");',
        '    String[] testConditions = config.getStringArray("testConditions");',
        '    String[] groupNames = config.getStringArray("gradeableLists");',
        '',
        f'    TestList testList = testListManager.create("{module_name}_workflow");',
        '',
        '    for (int i = 0; i < testConditions.length; i++) {',
        '      String condition = testConditions[i];',
        '      testList.setupBegin(condition)',
        '          .addAction(LevelChangeAction.class, "Begin_" + condition)',
        '          .setLevel(condition);',
        '',
        '      List<String> groupEntries = config.getStringList(groupNames[i]);',
        '      for (String entry : groupEntries) {',
        f'        String path = "{module_name}." + entry;',
        '        ConfigBlock testCaseConfig = ConfigLoader.load(paramFile, path);',
        '        String testCaseType = testCaseConfig.getString("testCase");',
        '        TestCaseBase testCase = testList.addTestCase(',
        '            Class.forName(testCaseType).asSubclass(TestCaseBase.class),',
        '            paramFile,',
        '            path',
        '        );',
        '        testCase.defineTestSequence();',
        '      }',
        '',
        '      testList.setupEnd(condition)',
        '          .addAction(LevelChangeAction.class, "End_" + condition)',
        '          .setLevel(endCondition);',
        '    }',
        '  }',
        '}',
    ]
    return '\n'.join(lines)

generated_code = generate_test_method('example_module')
print(generated_code)

## Step 4: Save The Artifact And Apply Quick Checks

Persisting the result to disk makes the workflow reviewable. A few lightweight checks then confirm that the generated class keeps the structure we expect before a human review or compilation step.

In [ ]:
output_dir = Path('../examples/generated_code')
output_dir.mkdir(exist_ok=True)

output_file = output_dir / 'ExampleGeneratedTestMethod.java'
output_file.write_text(generated_code)

checks = {
    'Has Javadoc': '/**' in generated_code,
    'No wildcard imports': '.*' not in generated_code,
    'Uses config loader': 'ConfigLoader.load' in generated_code,
    'Uses dynamic loading': 'Class.forName' in generated_code,
}

summary = {
    'generated_file': str(output_file),
    'line_count': len(generated_code.splitlines()),
    'quality_checks_passed': sum(checks.values()),
    'quality_checks_total': len(checks),
}

print(f'Saved to: {output_file}')
for name, passed in checks.items():
    print(f'{name}: {passed}')
for key, value in summary.items():
    print(f'{key}: {value}')

## Cleanup And References

Close the database client at the end so reruns stay predictable and independent.

Useful references:

- [README.md](../README.md) for the repository overview.
- [ARCHITECTURE.md](../docs/ARCHITECTURE.md) for the project structure.
- [MCP_INTEGRATION.md](../docs/MCP_INTEGRATION.md) for editor integration guidance.

In [ ]:
agent.close()
print('Notebook complete.')